# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bander03/FlyRank_Intern/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Reference: `docs/flyrank-seo-research-march-2026.pdf` ("The State of AI-Driven SEO," March 2026).

---

### Finding chosen #1 — ML Appendix, "What Predicts Health?" (page 27)

**The claim:** a Random Forest, holdout-tested, finds Average Position (43%) and Impressions (32%)
are the top two predictors of Health Score — 75% of total feature importance between just those
two inputs.

**My methodology question — where does the label come from?** The paper's own "Understanding the
Metrics" page (p.5) defines Health Score as `Impressions (30 pts) + position (30 pts) + CTR
(20 pts) + scroll depth (20 pts)` — a formula that *literally contains* two of the model's own
top features as additive terms. The paper is admirably upfront about this ("the target itself is
partly constructed from some of these inputs, so importance is descriptive rather than causal"),
which is exactly the right caveat to raise. My question extends it into something testable: **if
Impressions and Position are quite literally 60% of the formula's point allocation, would the
model's importance ranking collapse toward those two features simply because it's rediscovering
arithmetic, and if the two formula-derived features were removed entirely, would the remaining
model (Scroll Depth, CTR, Clicks, Sessions) retain enough signal to be useful on its own?** That's
a train-with/train-without test the paper doesn't show, and it's the standard check for whether
a "top predictor" is a genuine external pattern or a restated definition (see `SKILL.md`,
leakage taxonomy #1: "one feature towers over all others" is the symptom to test, not celebrate).
This doesn't mean the finding is wrong — the paper's own honest framing already suggests they know
this — it means the *causal-sounding phrasing* ("predicts health") could be tightened to "recovers
the components of its own definition," which is a smaller, more defensible claim.

---

### Finding chosen #2 — ML Appendix, "AI Model Performance" (page 16)

**The claim:** age-controlled cohort comparison, OpenAI vs. Gemini content, using Random Forest
and Logistic Regression validated with an "80/20 split" (per the Methodology page, p.36).

**My methodology question — does the validation design support the claim?** The paper covers 57
distinct brands, and the Methodology page doesn't specify whether that 80/20 split is a random
row-level split or a split grouped by brand/client. This matters a lot here specifically: if a
particular brand's editorial team standardized on one AI provider (plausible — many agencies pick
one model per client), then provider choice and brand identity would be correlated, and a random
split would let pages from the *same brand* appear in both the 80% training portion and the 20%
test portion. The model could then partly learn "brand X tends to score well" rather than "OpenAI
vs. Gemini content quality differs" — and brand-level effects (existing authority, editorial
process, topic mix) are exactly the kind of thing that would inflate an apparent provider
difference without it being a real provider effect. **My question, concretely: was the 80/20 split
grouped by brand, and if not, what does the same comparison look like under a brand-held-out
split?** The paper already draws the safe, narrow conclusion ("does not justify a blanket claim
that one model family universally wins") — this question is about whether the *evidence
underneath* that already-cautious conclusion is as solid as the caution suggests, not about
disputing the conclusion itself.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Week 5's model already used a client-grouped split. To show the before/after this section asks
for, I ran the *same* model (Random Forest, same features, same `low_ctr_for_type` label from
Week 5) under **two** splits on the same data: a naive random 75/25 row split (what a first-pass
notebook without this course's split discipline would likely do), and the honest client-grouped
split Week 5 actually used.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

RANDOM_SEED = 42
pd.set_option("display.width", 120)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
active = df[df["impressions_90d"] >= 100].copy().reset_index(drop=True)

num_feats = ["avg_position", "impressions_90d", "word_count", "content_age_days",
             "days_since_last_update", "engagement_rate", "scroll_rate"]
cat_feats = ["content_type"]
for f in num_feats:
    active[f] = active[f].fillna(0)
active["content_type"] = active["content_type"].fillna("unknown")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def label_and_run(train, test):
    """Same label + model as Week 5: low_ctr_for_type, RF, ctr excluded from features."""
    train, test = train.copy(), test.copy()
    train_type_p40 = train.groupby("content_type")["ctr"].quantile(0.40)
    global_p40 = train["ctr"].quantile(0.40)
    for frame in (train, test):
        thr = frame["content_type"].map(train_type_p40).fillna(global_p40)
        frame["label"] = (frame["ctr"] <= thr).astype(int)

    pre = ColumnTransformer([("num", "passthrough", num_feats),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats)])
    rf = Pipeline([("pre", pre), ("clf", RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, random_state=RANDOM_SEED))])
    rf.fit(train[num_feats + cat_feats], train["label"])
    proba = rf.predict_proba(test[num_feats + cat_feats])[:, 1]

    row = {"base_rate": round(test["label"].mean(), 3)}
    for k in (20, 50, 100):
        row[f"precision@{k}"] = round(precision_at_k(proba, test["label"].values, k), 3)
    row["client_overlap_train_test"] = len(set(train["client_id"]) & set(test["client_id"]))
    row["distinct_clients_in_test"] = test["client_id"].nunique()
    return row

# ---- BEFORE: naive random row-level 75/25 split (no grouping) ----
train_r, test_r = train_test_split(active, test_size=0.25, random_state=RANDOM_SEED)
before = label_and_run(train_r, test_r)

# ---- AFTER: honest client-grouped 75/25 split (same design as Week 5) ----
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
tr_idx, te_idx = next(gss.split(active, groups=active["client_id"]))
train_g, test_g = active.iloc[tr_idx], active.iloc[te_idx]
after = label_and_run(train_g, test_g)

comparison = pd.DataFrame([
    {"split": "BEFORE - naive random split", **before},
    {"split": "AFTER - honest client-grouped split", **after},
])
print(comparison.to_string(index=False))

                              split  base_rate  precision@20  precision@50  precision@100  client_overlap_train_test  distinct_clients_in_test
        BEFORE - naive random split      0.428           0.8          0.82           0.85                         28                        28
AFTER - honest client-grouped split      0.399           1.0          0.90           0.83                          0                         8


**Reading this honestly — and it's not the clean story I expected.** Precision doesn't move in
one consistent direction: the naive split scores *lower* at precision@20 (0.80 vs. 1.00) but
*higher* at precision@100 (0.85 vs. 0.83) than the honest split. If I only looked at the metric,
I could talk myself into either split looking "fine." **That's exactly why client overlap, not
the metric alone, is the real finding here:** the naive split has **28 of the same clients
appearing in both train and test** (out of the ~30 active clients in this data), while the
grouped split has **zero**. A precision number computed on a test set where 28 of ~30 clients
were already seen during training isn't measuring generalization to a new client — it's
measuring something closer to in-sample fit, whether or not the resulting number happens to look
higher or lower on a given run. The honest split's 0-overlap guarantee is the actual improvement;
the metric moving up or down is beside the point. This matches the skill's own instruction: "if
you can't explain the gap, you're not done" — here the gap isn't a clean inflation story, and
saying so is more honest than forcing one.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
# The attack checklist, applied to Week 5's final feature set:
print("Features used:", num_feats + cat_feats)
print("Label:", "low_ctr_for_type (ctr at/below the 40th pct for its own content_type)")
print()
print("[x] No label-derived columns in features -- ctr itself (which DEFINES the label) is")
print("    deliberately excluded, not just avoided by accident.")
print("[x] No trend_pct / trend_direction -- the OTHER label trap in this dataset (defines")
print("    is_declining_label elsewhere in the repo) -- neither is a feature here at all,")
print("    they aren't even relevant to this label, but confirmed absent.")
print("[x] No FlyRank product flags (health_score, needs_ctr_fix, is_quick_win) -- not present")
print("    in this dataset by design (see skills/flyrank/flyrank-context/SKILL.md).")
print("[x] Split grouped by client_id (Section 2 above).")
print("[x] Base rate printed next to every precision@K (Section 2 table).")
print()

# THE CONFESSION TEST: deliberately add ctr back in and watch the score jump
def label_and_run_with_ctr(train, test):
    train, test = train.copy(), test.copy()
    train_type_p40 = train.groupby("content_type")["ctr"].quantile(0.40)
    global_p40 = train["ctr"].quantile(0.40)
    for frame in (train, test):
        thr = frame["content_type"].map(train_type_p40).fillna(global_p40)
        frame["label"] = (frame["ctr"] <= thr).astype(int)
    feats = num_feats + ["ctr"]
    pre = ColumnTransformer([("num", "passthrough", feats),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats)])
    rf = Pipeline([("pre", pre), ("clf", RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, random_state=RANDOM_SEED))])
    rf.fit(train[feats + cat_feats], train["label"])
    proba = rf.predict_proba(test[feats + cat_feats])[:, 1]
    row = {"base_rate": round(test["label"].mean(), 3)}
    for k in (20, 50, 100):
        row[f"precision@{k}"] = round(precision_at_k(proba, test["label"].values, k), 3)
    return row

leaky = label_and_run_with_ctr(train_g, test_g)
confession = pd.DataFrame([
    {"feature_set": "LEAKY -- ctr added back in", **leaky},
    {"feature_set": "HONEST -- ctr excluded (Week 5 / Section 2 AFTER)", **after},
])
print(confession.to_string(index=False))

Features used: ['avg_position', 'impressions_90d', 'word_count', 'content_age_days', 'days_since_last_update', 'engagement_rate', 'scroll_rate', 'content_type']
Label: low_ctr_for_type (ctr at/below the 40th pct for its own content_type)

[x] No label-derived columns in features -- ctr itself (which DEFINES the label) is
    deliberately excluded, not just avoided by accident.
[x] No trend_pct / trend_direction -- the OTHER label trap in this dataset (defines
    is_declining_label elsewhere in the repo) -- neither is a feature here at all,
    they aren't even relevant to this label, but confirmed absent.
[x] No FlyRank product flags (health_score, needs_ctr_fix, is_quick_win) -- not present
    in this dataset by design (see skills/flyrank/flyrank-context/SKILL.md).
[x] Split grouped by client_id (Section 2 above).
[x] Base rate printed next to every precision@K (Section 2 table).



                                      feature_set  base_rate  precision@20  precision@50  precision@100  client_overlap_train_test  distinct_clients_in_test
                       LEAKY -- ctr added back in      0.399           1.0           1.0           1.00                        NaN                       NaN
HONEST -- ctr excluded (Week 5 / Section 2 AFTER)      0.399           1.0           0.9           0.83                        0.0                       8.0


**The confession, exactly as the skill predicts.** Adding `ctr` back as a feature pushes
precision to a flat **1.00 at every K** (20, 50, and 100) — "suspiciously perfect," the textbook
symptom of a label-derived feature, since `ctr` is literally what the label thresholds. Removing
it drops precision@100 back down to a believable 0.83. This confirms the test harness itself is
sound (it *can* detect leakage when leakage is deliberately present), and confirms Week 5's
honest number (0.83, not 1.00) is the real one to trust, not an artifact of an insufficiently
strict feature set.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Boldest sentence, from `w05_model.ipynb`, Section 3:**

> "Random Forest, given none of that direct signal, still ties the baseline at the top of the
> list (precision@20 = 1.00)."

Read plainly, this sounds like a settled property of the model. It isn't — it's one number from
one train/test split, on one 30k-row anonymized sample, with a label I defined myself (the 40th
percentile threshold). Section 2 of this notebook just demonstrated that precision@K on this
exact pipeline **moves around by split choice alone**, sometimes non-obviously. A sentence that
sounds causal/settled needs to say all of that.

**Rewritten, safe version:**

> Observed on this specific test split (client-grouped, 4,610 rows held out across 8 clients
> never seen in training): Random Forest's top-20 ranked list matched the baseline's precision@20
> exactly (1.00 vs. 1.00), even though the model had no access to the `ctr` value the baseline
> reads directly. This is a single-split, single-sample measurement — not a guaranteed or
> reproducible property of the model across all splits or data — and it should be read as
> directional decision-support for prioritizing review, not as proof the model has "solved" CTR
> risk detection. A model that would generalize this claim should show the same pattern averaged
> across several client-grouped splits, which this notebook has not yet done.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.